In [105]:
"""
Fighter Jet Classifier — EfficientNetV2S
=========================================
Upgrade from B0 → V2S:
  - Trained on ImageNet-21k (14M images, 21k classes) then fine-tuned on ImageNet-1k
  - 83.9% top-1 accuracy vs 77.1% for B0
  - include_preprocessing=True handles all pixel normalization inside the model
  - Native 384x384 resolution captures finer jet geometry details
 
Dataset structure:
  dataset/
    Train/      B2/ F22/ J20/ Rafale/ Su57/
    Validation/ B2/ F22/ J20/ Rafale/ Su57/
    Test/       B2/ F22/ J20/ Rafale/ Su57/
"""

'\nFighter Jet Classifier — EfficientNetV2S\n=========================================\nUpgrade from B0 → V2S:\n  - Trained on ImageNet-21k (14M images, 21k classes) then fine-tuned on ImageNet-1k\n  - 83.9% top-1 accuracy vs 77.1% for B0\n  - include_preprocessing=True handles all pixel normalization inside the model\n  - Native 384x384 resolution captures finer jet geometry details\n\nDataset structure:\n  dataset/\n    Train/      B2/ F22/ J20/ Rafale/ Su57/\n    Validation/ B2/ F22/ J20/ Rafale/ Su57/\n    Test/       B2/ F22/ J20/ Rafale/ Su57/\n'

In [106]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers, regularizers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# Config

IMG_SIZE        = (384, 384)   # Prev : (240,240)
BATCH_SIZE      = 8
NUM_CLASSES     = 5
CLASS_NAMES     = ["B2", "F22", "J20", "Rafale", "Su57"]

In [108]:
PHASE1_EPOCHS   = 20
PHASE2_EPOCHS   = 30
PHASE1_LR       = 1e-3
PHASE2_LR       = 1e-5

In [109]:
PATIENCE        = 10
SEED            = 42
MODEL_SAVE_PATH = "fighter_jet_classifier.h5"
DATASET_ROOT    = "data/Fighter_jet_final_data"  

In [110]:
tf.random.set_seed(SEED)
np.random.seed(SEED)

In [111]:
# DATA GENERATORS — NO rescale (model handles preprocessing)

In [112]:
train_datagen = ImageDataGenerator(
    rotation_range     = 20,
    width_shift_range  = 0.10,
    height_shift_range = 0.10,
    shear_range        = 0.10,
    zoom_range         = 0.15,
    horizontal_flip    = True,
    brightness_range   = [0.85, 1.15],
    fill_mode          = "nearest",
)

In [113]:
val_test_datagen = ImageDataGenerator()

In [114]:
train_gen = train_datagen.flow_from_directory(
    os.path.join(DATASET_ROOT, "Train"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "categorical",
    classes     = CLASS_NAMES,
    shuffle     = True,
    seed        = SEED,
)

Found 350 images belonging to 5 classes.


In [115]:
val_gen = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_ROOT, "Validation"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "categorical",
    classes     = CLASS_NAMES,
    shuffle     = False,
)

Found 50 images belonging to 5 classes.


In [116]:
test_gen = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_ROOT, "Test"),
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = "categorical",
    classes     = CLASS_NAMES,
    shuffle     = False,
)

Found 100 images belonging to 5 classes.


In [117]:
# DEBUG CHECKS

print("\n=== DEBUG CHECKS ===")
print(f"Class mapping   : {train_gen.class_indices}")
print(f"Train samples   : {train_gen.samples}  (expected 350)")
print(f"Val samples     : {val_gen.samples}    (expected 50)")
print(f"Test samples    : {test_gen.samples}   (expected 100)")
unique, counts = np.unique(train_gen.classes, return_counts=True)
print(f"Per-class counts: {dict(zip(unique, counts))}  (all should be 70)")
sample_batch = next(iter(train_gen))[0]
print(f"Pixel range     : min={sample_batch.min():.1f}  max={sample_batch.max():.1f}")
print(f"  -> Should be ~0 to ~255")
print("=== END DEBUG ===\n")


=== DEBUG CHECKS ===
Class mapping   : {'B2': 0, 'F22': 1, 'J20': 2, 'Rafale': 3, 'Su57': 4}
Train samples   : 350  (expected 350)
Val samples     : 50    (expected 50)
Test samples    : 100   (expected 100)
Per-class counts: {np.int32(0): np.int64(70), np.int32(1): np.int64(70), np.int32(2): np.int64(70), np.int32(3): np.int64(70), np.int32(4): np.int64(70)}  (all should be 70)
Pixel range     : min=0.0  max=255.0
  -> Should be ~0 to ~255
=== END DEBUG ===



In [118]:
# MODEL
# include_preprocessing=True means the model itself scales pixels
# correctly. Always feed raw 0-255 images everywhere.

In [119]:
def build_model() -> keras.Model:
    backbone = EfficientNetV2S(
        include_top           = False,
        weights               = "imagenet",
        input_shape           = (*IMG_SIZE, 3),
        include_preprocessing = True,   # handles all normalization internally
    )
    backbone.trainable = False
 
    inputs  = keras.Input(shape=(*IMG_SIZE, 3))
    x       = backbone(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(512, activation="relu",
                           kernel_regularizer=regularizers.l2(1e-4))(x)
    x       = layers.Dropout(0.5)(x)
    x       = layers.Dense(256, activation="relu",
                           kernel_regularizer=regularizers.l2(1e-4))(x)
    x       = layers.Dropout(0.4)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    return keras.Model(inputs, outputs)

In [120]:
def get_callbacks(suffix=""):
    return [
        callbacks.ModelCheckpoint(
            filepath=f"best_model{suffix}.h5",
            monitor="val_accuracy", save_best_only=True, verbose=1),
        callbacks.EarlyStopping(
            monitor="val_accuracy", patience=PATIENCE,
            restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=4, min_lr=1e-8, verbose=1),
    ]

In [122]:
# CLASS WEIGHTS

labels = train_gen.classes
cw = compute_class_weight("balanced", classes=np.unique(labels), y=labels)
class_weight_dict = dict(enumerate(cw))
print(f"Class weights: {class_weight_dict}\n")

Class weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0)}



In [123]:
print("=" * 55)
print("PHASE 1: Training classifier head (backbone frozen)")
print("=" * 55)

PHASE 1: Training classifier head (backbone frozen)


In [124]:
model = build_model()

82420632/82420632 ━━━━━━━━━━━━━━━━━━━━ 16s 0us/step


In [125]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=PHASE1_LR),
    loss="categorical_crossentropy", metrics=["accuracy"])

In [126]:
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 384, 384, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 12, 12, 1280)   │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,124,965 (80.59 MB)

 Trainable params: 791,045 (3.02 MB)

 Non-trainable params: 20,333,920 (77.57 MB)

In [127]:
history1 = model.fit(
    train_gen, 
    epochs=PHASE1_EPOCHS, 
    validation_data=val_gen,
    callbacks=get_callbacks("_phase1"), 
    class_weight=class_weight_dict, 
    verbose=1)

Epoch 1/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2898 - loss: 2.6056
Epoch 1: val_accuracy improved from None to 0.26000, saving model to best_model_phase1.h5



Epoch 1: finished saving model to best_model_phase1.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 98s 2s/step - accuracy: 0.3429 - loss: 2.6594 - val_accuracy: 0.2600 - val_loss: 1.5785 - learning_rate: 0.0010
Epoch 2/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4790 - loss: 1.9506
Epoch 2: val_accuracy improved from 0.26000 to 0.56000, saving model to best_model_phase1.h5



Epoch 2: finished saving model to best_model_phase1.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - accuracy: 0.4829 - loss: 1.9463 - val_accuracy: 0.5600 - val_loss: 1.4318 - learning_rate: 0.0010
Epoch 3/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5721 - loss: 1.5490
Epoch 3: val_accuracy did not improve from 0.56000
44/44 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - accuracy: 0.5600 - loss: 1.6314 - val_accuracy: 0.5000 - val_loss: 1.4266 - learning_rate: 0.0010
Epoch 4/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5122 - loss: 1.3979
Epoch 4: val_accuracy did not improve from 0.56000
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.5286 - loss: 1.4559 - val_accuracy: 0.5200 - val_loss: 1.3035 - learning_rate: 0.0010
Epoch 5/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6048 - loss: 1.3333
Epoch 5: val_accuracy did not improve from 0.56000
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.6200 - loss: 1.2535 - val_accuracy: 0.5200 - val_loss: 1.2083 - learning


Epoch 6: finished saving model to best_model_phase1.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 67s 2s/step - accuracy: 0.7057 - loss: 1.1148 - val_accuracy: 0.6200 - val_loss: 1.0697 - learning_rate: 0.0010
Epoch 7/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6634 - loss: 1.1448
Epoch 7: val_accuracy did not improve from 0.62000
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.6743 - loss: 1.1205 - val_accuracy: 0.5800 - val_loss: 1.2039 - learning_rate: 0.0010
Epoch 8/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7380 - loss: 0.8449
Epoch 8: val_accuracy did not improve from 0.62000
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.6886 - loss: 0.9326 - val_accuracy: 0.6000 - val_loss: 1.0903 - learning_rate: 0.0010
Epoch 9/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7151 - loss: 0.9534
Epoch 9: val_accuracy improved from 0.62000 to 0.74000, saving model to best_model_phase1.h5



Epoch 9: finished saving model to best_model_phase1.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.7200 - loss: 0.9218 - val_accuracy: 0.7400 - val_loss: 0.9602 - learning_rate: 0.0010
Epoch 10/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7380 - loss: 0.8989
Epoch 10: val_accuracy did not improve from 0.74000
44/44 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.7600 - loss: 0.8404 - val_accuracy: 0.7200 - val_loss: 0.8812 - learning_rate: 0.0010
Epoch 11/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7349 - loss: 0.9295
Epoch 11: val_accuracy did not improve from 0.74000
44/44 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.7486 - loss: 0.8843 - val_accuracy: 0.7400 - val_loss: 0.9817 - learning_rate: 0.0010
Epoch 12/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7830 - loss: 0.6968
Epoch 12: val_accuracy did not improve from 0.74000
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.7800 - loss: 0.6919 - val_accuracy: 0.7000 - val_loss: 1.0755 - le


Epoch 17: finished saving model to best_model_phase1.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.8486 - loss: 0.5521 - val_accuracy: 0.7600 - val_loss: 1.0927 - learning_rate: 5.0000e-04
Epoch 18/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8579 - loss: 0.5341
Epoch 18: val_accuracy did not improve from 0.76000

Epoch 18: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
44/44 ━━━━━━━━━━━━━━━━━━━━ 64s 1s/step - accuracy: 0.8914 - loss: 0.4537 - val_accuracy: 0.7200 - val_loss: 1.0177 - learning_rate: 5.0000e-04
Epoch 19/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9146 - loss: 0.4132
Epoch 19: val_accuracy did not improve from 0.76000
44/44 ━━━━━━━━━━━━━━━━━━━━ 65s 1s/step - accuracy: 0.8857 - loss: 0.5221 - val_accuracy: 0.6800 - val_loss: 1.0407 - learning_rate: 2.5000e-04
Epoch 20/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8072 - loss: 0.6816
Epoch 20: val_accuracy did not improve from 0.76000
44/44 ━━━━━━━━━━━━━━━━━━━━ 6

In [128]:
print("\n" + "=" * 55)
print("PHASE 2: Fine-tuning top-30 backbone layers")
print("=" * 55)


PHASE 2: Fine-tuning top-30 backbone layers


In [129]:
model.load_weights("best_model_phase1.h5")

In [130]:
backbone_layer = model.get_layer("efficientnetv2-s")
backbone_layer.trainable = True

In [131]:
for layer in backbone_layer.layers[:-30]:
    layer.trainable = False
for layer in backbone_layer.layers[-30:]:
    if not isinstance(layer, layers.BatchNormalization):
        layer.trainable = True

In [132]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=PHASE2_LR),
    loss="categorical_crossentropy", 
    metrics=["accuracy"])

In [133]:
history2 = model.fit(
    train_gen, 
    epochs=PHASE2_EPOCHS, 
    validation_data=val_gen,
    callbacks=get_callbacks("_phase2"), 
    class_weight=class_weight_dict, 
    verbose=1)

Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9092 - loss: 0.4387
Epoch 1: val_accuracy improved from None to 0.62000, saving model to best_model_phase2.h5



Epoch 1: finished saving model to best_model_phase2.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step - accuracy: 0.8829 - loss: 0.5084 - val_accuracy: 0.6200 - val_loss: 1.0933 - learning_rate: 1.0000e-05
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8965 - loss: 0.4772
Epoch 2: val_accuracy improved from 0.62000 to 0.66000, saving model to best_model_phase2.h5



Epoch 2: finished saving model to best_model_phase2.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 77s 2s/step - accuracy: 0.8800 - loss: 0.4857 - val_accuracy: 0.6600 - val_loss: 1.0128 - learning_rate: 1.0000e-05
Epoch 3/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8405 - loss: 0.5790
Epoch 3: val_accuracy improved from 0.66000 to 0.78000, saving model to best_model_phase2.h5



Epoch 3: finished saving model to best_model_phase2.h5
44/44 ━━━━━━━━━━━━━━━━━━━━ 75s 2s/step - accuracy: 0.8514 - loss: 0.5920 - val_accuracy: 0.7800 - val_loss: 0.9627 - learning_rate: 1.0000e-05
Epoch 4/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8901 - loss: 0.4863
Epoch 4: val_accuracy did not improve from 0.78000
44/44 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 0.8657 - loss: 0.5324 - val_accuracy: 0.7600 - val_loss: 1.0301 - learning_rate: 1.0000e-05
Epoch 5/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9045 - loss: 0.3666
Epoch 5: val_accuracy did not improve from 0.78000
44/44 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - accuracy: 0.8743 - loss: 0.4561 - val_accuracy: 0.7600 - val_loss: 0.9917 - learning_rate: 1.0000e-05
Epoch 6/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8361 - loss: 0.5301
Epoch 6: val_accuracy did not improve from 0.78000
44/44 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - accuracy: 0.8600 - loss: 0.5197 - val_accuracy: 0.7200 - val_loss: 1.009

In [134]:
print("\n" + "=" * 55)
print("FINAL EVALUATION ON TEST SET")
print("=" * 55)


FINAL EVALUATION ON TEST SET


In [135]:
model.load_weights("best_model_phase2.h5")
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"\nTest accuracy : {test_acc * 100:.2f}%")
print(f"Test loss     : {test_loss:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 17s 1s/step - accuracy: 0.7500 - loss: 0.8455

Test accuracy : 75.00%
Test loss     : 0.8455


In [136]:
test_gen.reset()
preds      = model.predict(test_gen, verbose=1)
pred_class = np.argmax(preds, axis=1)
true_class = test_gen.classes

13/13 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step


In [137]:
print("\nClassification Report:")
print(classification_report(true_class, pred_class, target_names=CLASS_NAMES))
cm = confusion_matrix(true_class, pred_class)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.close()
print("Saved → confusion_matrix.png")


Classification Report:
              precision    recall  f1-score   support

          B2       0.94      0.75      0.83        20
         F22       0.53      0.85      0.65        20
         J20       1.00      0.70      0.82        20
      Rafale       0.86      0.90      0.88        20
        Su57       0.65      0.55      0.59        20

    accuracy                           0.75       100
   macro avg       0.79      0.75      0.76       100
weighted avg       0.79      0.75      0.76       100

Saved → confusion_matrix.png


In [138]:
# HISTORY PLOT

acc      = history1.history["accuracy"]     + history2.history["accuracy"]
val_acc  = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss     = history1.history["loss"]         + history2.history["loss"]
val_loss = history1.history["val_loss"]     + history2.history["val_loss"]
ep       = range(1, len(acc) + 1)
p1_end   = len(history1.history["accuracy"])

In [139]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for ax, tv, vv, ylabel in [(ax1, acc, val_acc, "Accuracy"), (ax2, loss, val_loss, "Loss")]:
    ax.plot(ep, tv, label="Train",      color="#2980b9", linewidth=1.8)
    ax.plot(ep, vv, label="Validation", color="#e74c3c", linewidth=1.8)
    ax.axvline(x=p1_end, color="#7f8c8d", linestyle="--", linewidth=1, label="Phase 2 start")
    ax.set_xlabel("Epoch"); ax.set_ylabel(ylabel)
    ax.set_title(f"Training {ylabel}")
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("Fighter Jet Classifier (EfficientNetV2S) — Training History",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("training_history.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved → training_history.png")

Saved → training_history.png


In [140]:
# SAVE MODEL

model.save(MODEL_SAVE_PATH)
print(f"\nFinal model saved → {MODEL_SAVE_PATH}")
print("Done!")


Final model saved → fighter_jet_classifier.h5
Done!


In [141]:
# Test function

def predict_image(image_path: str, model_path: str = MODEL_SAVE_PATH) -> dict:
    """
    Predict a single image. Feed the raw image path —
    model handles all preprocessing internally.
 
    Example:
        result = predict_image("jet.jpg")
        print(result["predicted_class"], result["confidence"])
    """
    from tensorflow.keras.preprocessing import image as kimage
    mdl   = keras.models.load_model(model_path)
    img   = kimage.load_img(image_path, target_size=IMG_SIZE)
    arr   = np.expand_dims(kimage.img_to_array(img), axis=0)
    probs = mdl.predict(arr, verbose=0)[0]
    idx   = int(np.argmax(probs))
    return {
        "predicted_class" : CLASS_NAMES[idx],
        "confidence"      : float(probs[idx]),
        "all_scores"      : {c: float(p) for c, p in zip(CLASS_NAMES, probs)},
    }

In [143]:
predict_image('data/Testing_images/changchun-china-j-20-fighter-jet-performs-in-the-sky-during-the-2025-aviation-open-day.jpg')

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\Gandhi Uday\AppData\Local\Temp\ipykernel_16520\1694091636.py:1: SyntaxWarning: invalid escape sequence '\T'
  predict_image('data\Testing_images\changchun-china-j-20-fighter-jet-performs-in-the-sky-during-the-2025-aviation-open-day.jpg')


{'predicted_class': 'J20',
 'confidence': 0.9585797190666199,
 'all_scores': {'B2': 5.436653736978769e-05,
  'F22': 0.0007151245954446495,
  'J20': 0.9585797190666199,
  'Rafale': 0.038698792457580566,
  'Su57': 0.0019520543282851577}}

In [144]:
predict_image('data/Testing_images/rafale-2472066_640.jpg')

{'predicted_class': 'Rafale',
 'confidence': 0.9980397820472717,
 'all_scores': {'B2': 0.0007996429339982569,
  'F22': 5.501510531757958e-05,
  'J20': 5.200703890295699e-05,
  'Rafale': 0.9980397820472717,
  'Su57': 0.0010536344489082694}}